# pw03 — 우리 방법의 위치, 그리고 선행 방법론 수용

> ⚠ **이 노트북은 생성물이다. 수정은 `prior_work/src/make_pw03.py` 에서** 하고 재실행할 것.

pw01(논문)·pw02(도구)에서 확인한 선행을 바탕으로 **우리가 정확히 어디에 서 있는지**, **무엇이 주류와 같고 무엇이 덜 점유된 틈새인지**, 그리고 **선행 방법론을 우리 파이프라인에 어떻게 녹일지**를 정한다.

## §1. 우리는 주류 아키텍처 위에 있다 (이상하지 않다)

가장 자주 받은 의문 — *"σ 를 따로 구해 주입하는 게 편법 아닌가?"* — 에 대한 답은 **아니오, 그것이 표준이다**이다. 선행이 이를 뒷받침한다:

| 구성요소 | 우리 | 주류 선행 | 판정 |
|---|---|---|---|
| 표적/배경 채널 분리 `h = h_bg + h_target` | ✅ (report12) | NIST 5GNRad·3GPP ISAC·MATLAB | **동일** |
| 표적 밝기를 σ 로 채널에 주입 | ✅ (c) | NIST 5GNRad·MATLAB(MeanRCS) | **동일** |
| 환경 전파를 광선추적으로 담보 | ✅ Sionna RT | Deterministic-Modeling·SimART·ns3sionna | **동일** |
| 그 σ 를 **소형 드론 메쉬에서 SBR+PO 로 산출** | ✅ (d, report07) | 대개 확산계수 S 가정 or RCS 상수 | **차이(강화)** |

즉 우리 뼈대(분리채널+주입+RT담보)는 NIST 5GNRad·3GPP·MATLAB 과 **같은 주류 구조**다. 다른 점은 표적 밝기를 *가정*하지 않고 *계산*한다는 것 — 이건 약점이 아니라 강화다.

---
## §2. 덜 점유된 틈새 — 여기가 우리 기여

선행 지도를 겹쳐 보면 **세 축이 동시에 비어 있는 자리**에 우리가 있다:

1. **기회신호 패시브 바이스태틱** — 선행 Sionna-ISAC 은 대부분 **능동/모노스태틱**(NIST 5GNRad, MATLAB) 또는 **CSI 기반**(Great-X, ns3sionna)이다. 상용 WiFi/LTE/5G 를 *제어 없이 빌려* 기준·감시 채널로 쓰는 패시브 바이스태틱은 드물다.
2. **드론 RCS 를 확산계수 가정 대신 SBR+PO 로 메쉬에서 산출** — Great-X·Deterministic-Modeling 은 확산 S 를, NIST/MATLAB 은 RCS 상수를 쓴다. 우리는 재질·형상별 σ 를 광선조준+PO 적분으로 직접 낸다(report07/08).
3. **상시 vs 세션 9모드 벤치마크** — '패시브가 잠글 수 있는 기준신호'(WiFi 프리앰블·LTE CRS·5G SSB vs NR-PRS)로 탐지 난이도를 가르는 비교는 선행에서 못 봤다(report12).

> ⚠️ **과장 금지.** 각 축은 개별적으로는 선행이 있다(패시브 레이더 일반, 드론 RCS 측정, ISAC 벤치마크). 우리 기여는 **이 셋의 결합을 하나의 재현가능 파이프라인으로** 묶은 것이다.

---
## §3. 선행 방법론 수용 — 우리 파이프라인에 무엇을 녹이나

조사에서 배운 것을 **실제 작업으로** 옮긴다:

| 선행에서 배운 것 | 우리 반영 | 상태 |
|---|---|---|
| **표적/배경 채널 분리**(NIST 5GNRad·3GPP) 를 명시적 구조로 | report12 §2b 층별표에 `h=h_bg+h_target` 로 정식화, 유령=별도 target 탭 | 반영됨 |
| **확산계수 S 우회의 한계**(Deterministic-Modeling: 경면만이면 산란 과소평가) | report06 에 '왜 확산S 가정 대신 SBR+PO 인가' 근거로 인용 | 반영 예정 |
| **RadarSimPy 메쉬 RCS**(GPLv3) 로 독립 교차검증 | report07/08 σ·마이크로도플러를 RadarSimPy 로 재계산해 대조 | future work(도구 도입) |
| **OpenISAC OTA 바이스태틱 동기**(X410) | 실측 단계 골격으로 채택 — sim→real | future work(실측) |
| **NIST 5GNRad 검출체인**(range-Doppler·CFAR·clustering) | 우리 ECA→CAF→CFAR 설계 대조·용어 정렬 | 반영됨(설계 확인) |

이 표가 곧 **'선행 방법론을 프로젝트에 녹인다'**의 구체 목록이다. 반영된 것은 리포트 본문에, future work 는 report12 맺음말과 이 pw03 에 남긴다.

---
## §4. 정직한 한 문단 (외부에 말할 때)

> *"우리는 NVIDIA Sionna RT 로 챔버 전파를 담보하고(선행 Deterministic-Modeling·SimART 와 같은 용례), 표적 채널을 배경과 분리해 주입하는 주류 ISAC 아키텍처(NIST 5GNRad·3GPP)를 따른다. 차이는 표적 밝기 σ 를 확산계수로 가정하지 않고 소형 드론 메쉬에서 SBR+PO 로 산출한다는 것, 그리고 이를 상용 WiFi/LTE/5G 를 빌린 패시브 바이스태틱 구조에서 9모드로 벤치마크한다는 것이다. 검출체인(ECA·CAF·CFAR)은 어느 시뮬레이터도 제공하지 않는 연구 대상이라 직접 구현·검증했다."*

이 문단은 **모든 주장에 선행 근거**가 붙어 있어(pw01·pw02), 과장 없이 방어된다.

In [ ]:
# 조사 종합 — prior_work.json synthesis
import json
S = json.load(open('outputs/prior_work.json', encoding='utf-8'))['synthesis']
for k in ['q1_prior_sionna_isac', 'q2_gap_workarounds', 'our_position',
          'support_for_our_claim', 'adoption_plan']:
    print(f'■ {k}'); print('  ', S[k]); print()

---
## §5. 맺음

- **예 — 실재 확인. Great-X(드론·확산S), Deterministic-Modeling(차량·확산S·EuCAP), Ziganshin(차량·커스텀UTD), CISSIR(NVIDIA공식·표적없음), SimART(비전센싱). Sionna 를 ISAC 에 쓴 선행은 다수다.**
- **우리는 (d)로 값을 계산(SBR+PO)해 (c)로 채널에 주입 — h=h_bg+h_target 는 NIST 5GNRad·3GPP 와 같은 주류 아키텍처. 덜 점유된 틈새: ①기회신호(WiFi/LTE/5G) 패시브 바이스태틱(대개 능동/모노 또는 CSI기반) ②드론 RCS 를 확산S 가정 대신 SBR+PO 로 메쉬에서 산출 ③상시vs세션 9모드 벤치마크.**

조사 원문 요약은 `/data/public/sionna_jeong/papers_isac_sionna/`(지속 확인용)와 이 폴더의 `prior_work.json` 에 보존한다. 새 선행이 나오면 JSON 에 추가하고 세 리포트를 재빌드하면 된다.